# Subcluster Cards — `01b_subcluster_cards.ipynb`

Generate an **interactive HTML report** and a **PowerPoint deck** (one slide per subcluster) from the outputs of `01_disease_subcluster_detection.ipynb`.

Run this notebook **once per tissue**, using the same `RESULTS_DIR` you used in Notebook 1.

**Outputs (all written to `{OUTPUT_DIR}/{TISSUE}_subcluster_report/`):**
- `index.html` — interactive offline HTML with sidebar navigation, UMAP panels, volcano/dotplot, and paginated DE table (DataTables)
- `img/` — PNG files referenced by the HTML
- `de/` — per-subcluster DE CSV files (significant genes, downloadable from the report)
- `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` — static PowerPoint deck


## 1. Imports

In [ ]:
from __future__ import annotations

import base64
import io
import json
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from jinja2 import Environment, FileSystemLoader
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm, Inches, Pt

from scutils.plotting.volcano_plot import volcano_plot
from scutils.plotting.functional import create_pathway_dotplot
from scutils.tools.differential_expression import format_deseq2_results

matplotlib.use("Agg")
warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports OK.")


## 2. Configuration

Key parameters:
- `ADATA_PATH` — `.h5ad` file used in Notebook 1 (required for UMAP panels)
- `RESULTS_DIR` — output directory from Notebook 1 for this tissue
- `TISSUE` — tissue/dataset label (used in filenames and report headers)
- `SUBCLUSTER_KEY` — `adata.obs` column that holds subcluster labels (default: `"disease_subcluster"`)
- `DISEASE_KEY` / `CELLTYPE_KEY` / `CONDITION_KEY` — `adata.obs` columns
- `CONTROL_LABEL` — value of `CONDITION_KEY` that represents healthy/control
- `OUTPUT_DIR` — where HTML and PPTX are saved (defaults to `RESULTS_DIR`)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
ADATA_PATH    : Path = Path("data/my_tissue.h5ad")
RESULTS_DIR   : Path = Path("results/disease_subclusters/my_tissue")
TISSUE        : str  = "my_tissue"

# ── AnnData keys ──────────────────────────────────────────────────────────────
SUBCLUSTER_KEY : str = "disease_subcluster"
CELLTYPE_KEY   : str = "cell_type"
DISEASE_KEY    : str = "disease"          # column with disease/condition labels
CONDITION_KEY  : str = "disease"          # column used to split healthy vs disease
CONTROL_LABEL  : str = "control"          # value in CONDITION_KEY for healthy cells

# ── DE columns expected in the CSV (must match Notebook 1 output) ─────────────
DE_PVAL_COL     : str = "padj"
DE_LFC_COL      : str = "log2FoldChange"
GENE_SYMBOL_COL : str = "gene"            # column with gene symbols in DE CSV

# ── Volcano plot settings ─────────────────────────────────────────────────────
VOLCANO_PVAL_CUTOFF : float = 0.05
VOLCANO_LFC_CUTOFF  : float = 0.5
VOLCANO_TOP_N_UP    : int   = 10
VOLCANO_TOP_N_DOWN  : int   = 5
VOLCANO_FIGSIZE     : tuple = (10, 6)

# ── Enrichment dotplot settings ───────────────────────────────────────────────
ENRICHMENT_MAX_PATHWAYS : int   = 15
ENRICHMENT_FIGSIZE      : tuple = (12, 6)
PATHWAY_COLORS          : dict  = {
    "GO:BP": "#1f77b4",
    "GO:MF": "#ff7f0e",
    "GO:CC": "#2ca02c",
    "KEGG":  "#d62728",
    "REAC":  "#9467bd",
    "WP":    "#CD853F",
}

# ── UMAP settings ─────────────────────────────────────────────────────────────
UMAP_FIGSIZE        : tuple = (6, 5)
UMAP_POINT_SIZE     : float = None   # None = auto
BACKGROUND_COLOR    : str   = "#d3d3d3"

# ── Output ────────────────────────────────────────────────────────────────────
# HTML report is written to a subdirectory for lightweight file-based access.
#   {OUTPUT_DIR}/{TISSUE}_subcluster_report/
#       index.html       ← open this in a browser
#       img/             ← PNG files referenced by the HTML
OUTPUT_DIR : Path = RESULTS_DIR   # parent directory for all outputs

# ── PPTX layout ───────────────────────────────────────────────────────────────
PPTX_TOP_DE_GENES   : int = 20   # max rows in the DE table (upregulated only)

# ─────────────────────────────────────────────────────────────────────────────
# Build report directory structure
REPORT_DIR     : Path = OUTPUT_DIR / f"{TISSUE}_subcluster_report"
REPORT_IMG_DIR : Path = REPORT_DIR / "img"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_IMG_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"  Tissue      : {TISSUE}")
print(f"  Results dir : {RESULTS_DIR}")
print(f"  Report dir  : {REPORT_DIR}")
print(f"  Image dir   : {REPORT_IMG_DIR}")


## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded: {adata}")

# Validate required keys
for key, attr in [
    (SUBCLUSTER_KEY, "obs"),
    (CELLTYPE_KEY,   "obs"),
    (DISEASE_KEY,    "obs"),
]:
    assert key in getattr(adata, attr).columns, (
        f"Key '{key}' not found in adata.{attr}. "
        f"Available: {list(getattr(adata, attr).columns)}"
    )

info_key = f"{SUBCLUSTER_KEY}_info"
assert info_key in adata.uns, (
    f"'{info_key}' not found in adata.uns. "
    "Ensure you ran 01_disease_subcluster_detection.ipynb first."
)

subcluster_info: pd.DataFrame = adata.uns[info_key].copy()
background_mask = adata.obs[SUBCLUSTER_KEY] == "background"

# All non-background subcluster labels
subcluster_labels = [
    lbl for lbl in adata.obs[SUBCLUSTER_KEY].unique()
    if lbl != "background"
]
print(f"Found {len(subcluster_labels)} disease-enriched subclusters:")
for lbl in sorted(subcluster_labels):
    print(f"  {lbl}")


## 4. Helper Functions

In [ ]:
import re as _re

def _fig_to_b64(fig: plt.Figure) -> str:
    """Encode a matplotlib Figure as a base64 PNG string."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


def _fig_to_bytes(fig: plt.Figure) -> bytes:
    """Return a matplotlib Figure as PNG bytes."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    return buf.read()


def _safe_label(label: str) -> str:
    """Convert a subcluster label to a filesystem-safe, URL-safe string.

    Replaces every character that is not alphanumeric, a hyphen, or an
    underscore with ``_``, then collapses consecutive underscores.
    This ensures no special characters (``+``, ``'``, ``:``, ``(``, etc.)
    appear in filenames or HTML ``src`` attributes.
    """
    safe = _re.sub(r'[^\w\-]', '_', label)
    safe = _re.sub(r'_+', '_', safe)
    return safe.strip('_')


def _make_umap_celltype(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by cell type."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=CELLTYPE_KEY, ax=ax, show=False,
               frameon=False, title="Cell type")
    return fig


def _make_umap_disease(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by disease condition."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=DISEASE_KEY, ax=ax, show=False,
               frameon=False, title="Disease")
    return fig


def _make_umap_highlight(
    adata: AnnData,
    subcluster_label: str,
    figsize: tuple,
) -> plt.Figure:
    """UMAP with subcluster cells highlighted; background cells in grey."""
    mask = adata.obs[SUBCLUSTER_KEY] == subcluster_label

    fig, ax = plt.subplots(figsize=figsize)
    coords = adata.obsm["X_umap"]
    pt_size = UMAP_POINT_SIZE or max(120000 / adata.n_obs, 0.5)
    ax.scatter(
        coords[~mask, 0], coords[~mask, 1],
        s=pt_size, c=BACKGROUND_COLOR, linewidths=0, rasterized=True,
    )
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        s=pt_size * 1.5, c="#e63946", linewidths=0, rasterized=True, zorder=2,
    )
    ax.set_title(f"Subcluster: {subcluster_label}", fontsize=10)
    ax.axis("off")
    return fig


def _fmt_float(v, decimals: int = 3) -> str:
    try:
        return f"{float(v):.{decimals}g}"
    except (ValueError, TypeError):
        return str(v)


def _parse_stats(label: str, info_df: pd.DataFrame) -> dict:
    """Extract per-subcluster stats using the actual subcluster_info column names."""
    row = info_df[info_df["label"] == label]
    if row.empty:
        return {}
    row = row.iloc[0]

    n_total   = int(row["n_cells"])         if pd.notna(row.get("n_cells"))         else "—"
    n_disease = int(row["n_disease_cells"]) if pd.notna(row.get("n_disease_cells")) else "—"
    n_healthy = (
        n_total - n_disease
        if isinstance(n_total, int) and isinstance(n_disease, int)
        else "—"
    )

    # disease_composition: prefer dedicated column (COMBINE_DISEASES='pool'),
    # then diseases_contributing string, then fall back to disease label
    if "disease_composition" in info_df.columns and pd.notna(row.get("disease_composition")):
        disease_composition = str(row["disease_composition"])
    else:
        diseases_str = (
            row.get("diseases_contributing", "")
            if "diseases_contributing" in info_df.columns
            else ""
        )
        if diseases_str and str(diseases_str).strip():
            disease_composition = str(diseases_str)
        else:
            disease_composition = str(row.get("disease", "—"))

    return {
        "n_total":             n_total,
        "n_disease":           n_disease,
        "n_healthy":           n_healthy,
        "fold_enrichment":     _fmt_float(row.get("fold_enrichment", "—")),
        "fdr_pval":            _fmt_float(row.get("pvalue_adj", "—"), decimals=2),
        "cell_type":           str(row.get("subset", "—")),
        "disease_composition": disease_composition,
    }


def _parse_disease_breakdown(
    label: str,
    info_df: pd.DataFrame,
    adata: AnnData,
) -> list[dict]:
    """
    Build a per-disease breakdown for a subcluster.

    When combine_diseases is not None, `diseases_contributing` contains a
    comma-separated string of actual disease labels. Per-disease cell counts
    are computed directly from adata.obs so we don't need extra columns.
    When combine_diseases is None, the subcluster is already disease-specific
    (disease column = actual disease name) — return a single-row breakdown.
    """
    row = info_df[info_df["label"] == label]
    if row.empty:
        return []
    row = row.iloc[0]

    subcluster_mask = adata.obs[SUBCLUSTER_KEY] == label
    n_subcluster = subcluster_mask.sum()
    if n_subcluster == 0:
        return []

    # Case 1: diseases_contributing column present and populated
    diseases_str = row.get("diseases_contributing", "") if "diseases_contributing" in info_df.columns else ""
    if diseases_str and str(diseases_str).strip():
        diseases = [d.strip() for d in str(diseases_str).split(",") if d.strip()]
        breakdown = []
        for d in sorted(diseases):
            disease_cells_mask = (
                subcluster_mask & (adata.obs[DISEASE_KEY] == d)
            )
            n = int(disease_cells_mask.sum())
            pct = f"{100 * n / n_subcluster:.1f}%"
            breakdown.append({
                "disease": d,
                "n_cells": n,
                "pct":     pct,
                "fold":    "—",
                "pval":    "—",
                "fdr":     "—",
            })
        return breakdown

    # Case 2: single-disease subcluster (combine_diseases=None / "union")
    disease_label = str(row.get("disease", ""))
    if disease_label and disease_label != "combined":
        n = int(row.get("n_disease_cells", 0))
        pct = f"{100 * n / n_subcluster:.1f}%" if n_subcluster > 0 else "—"
        return [{
            "disease": disease_label,
            "n_cells": n,
            "pct":     pct,
            "fold":    _fmt_float(row.get("fold_enrichment", "—")),
            "pval":    _fmt_float(row.get("pvalue", "—")),
            "fdr":     _fmt_float(row.get("pvalue_adj", "—"), decimals=2),
        }]

    return []


def _load_de(label: str) -> pd.DataFrame | None:
    """
    Load DE results CSV for a subcluster.

    Notebook 1 saves comparisons as:
        de/de_{safe_comparison_label}.csv
    where the comparison label starts with the safe subcluster label
    (e.g. de_Principal_cell_combined_sub1_vs_Principal_cell_Control.csv).

    We glob for any file whose name contains the safe subcluster label
    and merge results if multiple comparisons exist.
    """
    safe = _safe_label(label)
    de_dir = RESULTS_DIR / "de"
    if not de_dir.exists():
        print(f"  [warn] DE directory not found: {de_dir}")
        return None

    matches = [
        p for p in sorted(de_dir.glob(f"de_{safe}*.csv"))
        if "all_comparisons" not in p.name
    ]
    if not matches:
        matches = [
            p for p in sorted(de_dir.glob("de_*.csv"))
            if safe in p.stem and "all_comparisons" not in p.name
        ]
    if not matches:
        print(f"  [warn] No DE file found for '{label}' in {de_dir}")
        return None

    frames = []
    for p in matches:
        df = pd.read_csv(p)
        df["_comparison"] = p.stem.removeprefix("de_")
        frames.append(df)

    if len(frames) == 1:
        return frames[0].drop(columns=["_comparison"])
    combined = pd.concat(frames, ignore_index=True)
    cols = ["_comparison"] + [c for c in combined.columns if c != "_comparison"]
    return combined[cols]


def _load_enrichment(label: str) -> pd.DataFrame | None:
    """
    Load enrichment CSV for a subcluster.

    Notebook 1 saves enrichment as:
        enrichment/enrichment_{safe_comparison_label}.csv
    We glob for any file whose name contains the safe subcluster label.
    """
    safe = _safe_label(label)
    enrich_dir = RESULTS_DIR / "enrichment"
    if not enrich_dir.exists():
        print(f"  [warn] Enrichment directory not found: {enrich_dir}")
        return None

    matches = [
        p for p in sorted(enrich_dir.glob(f"enrichment_{safe}*.csv"))
        if "all_comparisons" not in p.name
    ]
    if not matches:
        matches = [
            p for p in sorted(enrich_dir.glob("enrichment_*.csv"))
            if safe in p.stem and "all_comparisons" not in p.name
        ]
    if not matches:
        print(f"  [warn] No enrichment file found for '{label}' in {enrich_dir}")
        return None

    frames = [pd.read_csv(p) for p in matches]
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]


# Preferred display columns for DE table (in order of preference)
_DE_DISPLAY_COLS = [
    "_comparison", GENE_SYMBOL_COL, "gene_id",
    DE_LFC_COL, "logfoldchanges",
    DE_PVAL_COL, "pvals_adj",
    "pvalue", "pvals",
    "baseMean", "stat",
]


_PVAL_DISPLAY_COLS = {"padj", "pvalue", "pvals_adj", "pvals", DE_PVAL_COL}


def _de_display_df(de_df: pd.DataFrame) -> pd.DataFrame:
    """Return a display-ready DE DataFrame with gene symbols first.

    P-value columns are formatted in scientific notation (e.g. 1.23e-200)
    so that very small values are not silently rounded to 0.
    """
    show_cols = [c for c in _DE_DISPLAY_COLS if c in de_df.columns]
    if not show_cols:
        show_cols = list(de_df.columns[:8])
    out = de_df[show_cols].copy()
    for c in out.select_dtypes(include="float").columns:
        if c in _PVAL_DISPLAY_COLS:
            out[c] = out[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "")
        else:
            out[c] = out[c].round(4)
    return out


def _round_df(df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """Round float columns for display."""
    out = df.copy()
    for col in out.select_dtypes(include="float").columns:
        out[col] = out[col].round(decimals)
    return out


print("Helper functions defined.")


## 5. Generate Per-Subcluster Assets

In [ ]:
subcluster_data = []  # list of dicts, one per subcluster

for label in sorted(subcluster_labels):
    safe = _safe_label(label)
    print(f"\n── {label}")

    card = {
        "label":             label,
        "anchor":            safe,
        "disease":           label.split("|")[1] if "|" in label else label,
        "stats":             _parse_stats(label, subcluster_info),
        "disease_breakdown": _parse_disease_breakdown(label, subcluster_info, adata),
    }

    # Rebuild disease_composition from the per-disease breakdown when the
    # stored value is a generic placeholder (e.g. "combined").
    _bd = card["disease_breakdown"]
    if _bd:
        _comp = card["stats"].get("disease_composition", "")
        if not _comp or _comp.lower() in ("combined", "—", "unknown", "nan"):
            card["stats"]["disease_composition"] = ", ".join(
                f"{r['disease']}: {r['n_cells']} ({r['pct']})" for r in _bd
            )

    # ── UMAP panels ──────────────────────────────────────────────────────
    print("  generating UMAP panels...", end=" ")
    try:
        fig_ct  = _make_umap_celltype(adata, UMAP_FIGSIZE)
        fig_dis = _make_umap_disease(adata, UMAP_FIGSIZE)
        fig_hl  = _make_umap_highlight(adata, label, UMAP_FIGSIZE)

        for key, fig, suffix in [
            ("umap_celltype",  fig_ct,  "umap_celltype"),
            ("umap_disease",   fig_dis, "umap_disease"),
            ("umap_highlight", fig_hl,  "umap_highlight"),
        ]:
            img_name = f"{safe}_{suffix}.png"
            fig.savefig(REPORT_IMG_DIR / img_name, bbox_inches="tight", dpi=150)
            card[f"{key}_path"]  = f"img/{img_name}"
            card[f"{key}_bytes"] = _fig_to_bytes(fig)
            plt.close(fig)

        print("OK")
    except Exception as e:
        print(f"FAILED ({e})")
        for key in ["umap_celltype", "umap_disease", "umap_highlight"]:
            card[f"{key}_path"]  = None
            card[f"{key}_bytes"] = None

    # ── Load DE results ───────────────────────────────────────────────────
    de_df = _load_de(label)

    # ── Volcano plot ──────────────────────────────────────────────────────
    # Use format_deseq2_results so gene symbols are annotated identically
    # to Notebook 01 (set gene column as index → names column).
    print("  generating volcano plot...", end=" ")
    if de_df is not None and GENE_SYMBOL_COL in de_df.columns:
        try:
            df_volcano = format_deseq2_results(
                de_df.set_index(GENE_SYMBOL_COL),
                pval_col=DE_PVAL_COL,
                lfc_col=DE_LFC_COL,
            )
            fig_vol = volcano_plot(
                df=df_volcano,
                pval_cutoff=VOLCANO_PVAL_CUTOFF,
                lfc_cutoff=VOLCANO_LFC_CUTOFF,
                top_n_up=VOLCANO_TOP_N_UP,
                top_n_down=VOLCANO_TOP_N_DOWN,
                figsize=VOLCANO_FIGSIZE,
            )
            img_name = f"{safe}_volcano.png"
            fig_vol.savefig(REPORT_IMG_DIR / img_name, bbox_inches="tight", dpi=150)
            card["volcano_path"]  = f"img/{img_name}"
            card["volcano_bytes"] = _fig_to_bytes(fig_vol)
            plt.close(fig_vol)
            print("OK")
        except Exception as e:
            print(f"FAILED ({e})")
            card["volcano_path"]  = None
            card["volcano_bytes"] = None
    elif de_df is not None:
        try:
            fig_vol = volcano_plot(
                df=de_df,
                pval_col=DE_PVAL_COL,
                lfc_col=DE_LFC_COL,
                pval_cutoff=VOLCANO_PVAL_CUTOFF,
                lfc_cutoff=VOLCANO_LFC_CUTOFF,
                top_n_up=VOLCANO_TOP_N_UP,
                top_n_down=VOLCANO_TOP_N_DOWN,
                figsize=VOLCANO_FIGSIZE,
            )
            img_name = f"{safe}_volcano.png"
            fig_vol.savefig(REPORT_IMG_DIR / img_name, bbox_inches="tight", dpi=150)
            card["volcano_path"]  = f"img/{img_name}"
            card["volcano_bytes"] = _fig_to_bytes(fig_vol)
            plt.close(fig_vol)
            print("OK (no gene symbols)")
        except Exception as e:
            print(f"FAILED ({e})")
            card["volcano_path"]  = None
            card["volcano_bytes"] = None
    else:
        card["volcano_path"]  = None
        card["volcano_bytes"] = None
        print("skipped (no DE data)")

    # ── DE DataFrame (stored for HTML render cell and PPTX) ───────────────
    card["de_df"] = de_df

    # ── Enrichment dotplot ────────────────────────────────────────────────
    print("  generating enrichment dotplot...", end=" ")
    enrich_df = _load_enrichment(label)
    if enrich_df is not None:
        try:
            fig_dot = create_pathway_dotplot(
                data=enrich_df,
                source_colors=PATHWAY_COLORS,
                figsize=ENRICHMENT_FIGSIZE,
                max_pathways=ENRICHMENT_MAX_PATHWAYS,
                title=f"Functional enrichment — {label}",
            )
            img_name = f"{safe}_enrichment.png"
            fig_dot.savefig(REPORT_IMG_DIR / img_name, bbox_inches="tight", dpi=150)
            card["dotplot_path"]  = f"img/{img_name}"
            card["dotplot_bytes"] = _fig_to_bytes(fig_dot)
            plt.close(fig_dot)
            print("OK")
        except Exception as e:
            print(f"FAILED ({e})")
            card["dotplot_path"]  = None
            card["dotplot_bytes"] = None
    else:
        card["dotplot_path"]  = None
        card["dotplot_bytes"] = None

    subcluster_data.append(card)

print(f"\n✓ Assets generated for {len(subcluster_data)} subclusters.")
print(f"  Images saved to: {REPORT_IMG_DIR}")


## 6. Render HTML Report

In [ ]:
TEMPLATE_DIR = Path(__file__).parent / "templates" if "__file__" in dir() else \
               Path("notebooks/disease_subclusters/templates")

# ── Save DE CSVs and prepare per-card JSON for the lazy DE table ──────────────
# CSVs are saved alongside the report so users can open / download raw data.
# The same data is embedded as JSON so the table works entirely offline from
# file://, with no external dependencies.
de_csv_dir = REPORT_DIR / "de"
de_csv_dir.mkdir(exist_ok=True)

for card in subcluster_data:
    de_df = card.get("de_df")
    if de_df is not None and not de_df.empty:
        # Filter to significant genes; fall back to all if none pass
        if DE_PVAL_COL in de_df.columns:
            sig_df = de_df[de_df[DE_PVAL_COL] < VOLCANO_PVAL_CUTOFF].copy()
        else:
            sig_df = de_df.copy()
        display_df = sig_df if not sig_df.empty else de_df.copy()

        # Sort: most upregulated (highest LFC) first, then by p-value ascending
        sort_keys, sort_asc = [], []
        if DE_LFC_COL in display_df.columns:
            sort_keys.append(DE_LFC_COL);  sort_asc.append(False)
        if DE_PVAL_COL in display_df.columns:
            sort_keys.append(DE_PVAL_COL); sort_asc.append(True)
        if sort_keys:
            display_df = display_df.sort_values(sort_keys, ascending=sort_asc)

        # Save CSV for the download link in the report
        csv_path = de_csv_dir / f"{card['anchor']}.csv"
        _de_display_df(display_df).to_csv(csv_path, index=False)
        card["de_csv_path"] = f"de/{card['anchor']}.csv"

        # Serialise as {"columns": [...], "data": [[...],...]} for the inline table
        de_display = _de_display_df(display_df)
        card["de_json"] = de_display.to_json(orient="split")
        n_sig = len(sig_df)
    else:
        card["de_json"]     = None
        card["de_csv_path"] = None
        n_sig = 0

    print(f"  {card['label']}: {n_sig} significant genes in HTML table")

# ── Render template ────────────────────────────────────────────────────────────
env      = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)), autoescape=False)
template = env.get_template("subcluster_report.html.j2")

html_out = template.render(
    tissue=TISSUE,
    subclusters=subcluster_data,
    generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
)

html_path = REPORT_DIR / "index.html"
html_path.write_text(html_out, encoding="utf-8")
size_kb = html_path.stat().st_size / 1024
print(f"\n✓ HTML report written: {html_path}  ({size_kb:.0f} KB)")
print(f"  DE CSVs written to : {de_csv_dir}")
print(f"  Open in browser    : {html_path.resolve()}")


## 7. Render PowerPoint Deck

In [ ]:

# ── Layout constants ─────────────────────────────────────────────────────────
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

_DARK  = RGBColor(0x1a, 0x1a, 0x2e)
_BLUE  = RGBColor(0x4a, 0x90, 0xd9)
_WHITE = RGBColor(0xff, 0xff, 0xff)
_LGREY = RGBColor(0xf0, 0xf2, 0xf5)
_TEXT  = RGBColor(0x1a, 0x1a, 0x2e)

# Left panel (stats + DE table)
_LEFT_L = Cm(0.4)
_LEFT_W = Cm(12.7)
# Right panel (images)
_RIGHT_L = Cm(13.5)


def _add_text_box(slide, left, top, width, height, text, *, bold=False,
                  fontsize=10, color=_TEXT, align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf    = txBox.text_frame
    tf.word_wrap = wrap
    p      = tf.paragraphs[0]
    p.text = text
    p.alignment = align
    run = p.runs[0] if p.runs else p.add_run()
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color
    return txBox


def _add_image_bytes(slide, img_bytes, left, top, width, height):
    if img_bytes is None:
        return
    buf = io.BytesIO(img_bytes)
    slide.shapes.add_picture(buf, left, top, width=width, height=height)


def _add_title_bar(slide, label, disease):
    bar = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        0, 0, SLIDE_W, Cm(1.8),
    )
    bar.fill.solid()
    bar.fill.fore_color.rgb = _DARK
    bar.line.fill.background()

    _add_text_box(
        slide, Cm(0.4), Cm(0.1), Cm(20), Cm(1.6),
        label, bold=True, fontsize=14, color=_WHITE,
    )
    _add_text_box(
        slide, Cm(21), Cm(0.1), Cm(12), Cm(1.6),
        f"Disease: {disease}", bold=False, fontsize=11, color=_BLUE,
        align=PP_ALIGN.RIGHT,
    )


def _set_cell_text(cell, text: str, *, bold: bool = False, fontsize: int = 6,
                   bg_color: RGBColor | None = None,
                   color: RGBColor = _TEXT):
    """Set text and font on a pptx table cell (explicit color prevents theme inheritance)."""
    if bg_color is not None:
        cell.fill.solid()
        cell.fill.fore_color.rgb = bg_color
    tf = cell.text_frame
    tf.word_wrap = False
    p = tf.paragraphs[0]
    p.clear()
    run = p.add_run()
    run.text = text
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color


def _add_table_section_title(slide, left, top, width, text):
    """Add a small uppercase section label above a table; returns new top."""
    _add_text_box(slide, left, top, width, Cm(0.42),
                  text, bold=True, fontsize=7, color=_BLUE)
    return top + Cm(0.42)


def _add_stats_table(slide, stats, breakdown, top=Cm(2.0),
                     left=Cm(0.4), width=Cm(12.7)):
    """Render subcluster stats as a two-column pptx table with a labelled title.

    Always shows the disease_composition string (rebuilt from breakdown in
    the asset loop). Per-disease breakdown rows are appended below it.
    Returns the bottom y-coordinate so the next element can be placed below.
    """
    rows_data = [
        ("Total cells",        str(stats.get("n_total",         "—"))),
        ("Disease cells",      str(stats.get("n_disease",       "—"))),
        ("Healthy cells",      str(stats.get("n_healthy",       "—"))),
        ("Fold enrichment",    str(stats.get("fold_enrichment", "—"))),
        ("FDR p-value",        str(stats.get("fdr_pval",        "—"))),
        ("Cell type",          str(stats.get("cell_type",       "—"))),
        ("Disease composition",str(stats.get("disease_composition", "—"))),
    ]
    # Per-disease breakdown rows (indented)
    if breakdown:
        for r in breakdown:
            rows_data.append((
                f"  \u21b3 {r['disease']}",
                f"n={r['n_cells']} ({r['pct']})"
                + (f"  FDR={r['fdr']}" if r['fdr'] != "—" else ""),
            ))

    tbl_top = _add_table_section_title(slide, left, top, width, "SUMMARY STATISTICS")
    row_h   = Cm(0.42)
    n_rows  = len(rows_data)
    tbl     = slide.shapes.add_table(n_rows, 2, left, tbl_top, width, row_h * n_rows).table

    tbl.columns[0].width = Cm(4.5)
    tbl.columns[1].width = width - Cm(4.5)

    for ri, (lbl, val) in enumerate(rows_data):
        _set_cell_text(tbl.cell(ri, 0), lbl, bold=True,  fontsize=6, bg_color=_LGREY)
        _set_cell_text(tbl.cell(ri, 1), val, bold=False, fontsize=6)

    return tbl_top + row_h * n_rows   # bottom y


def _add_de_table(slide, de_df, top, left=Cm(0.4), width=Cm(12.7)):
    """Render top-N significant upregulated DE genes as a pptx table with title."""
    if de_df is None or de_df.empty:
        return

    # Filter to significant upregulated genes only
    up_df = de_df.copy()
    if DE_PVAL_COL in up_df.columns:
        up_df = up_df[up_df[DE_PVAL_COL] < VOLCANO_PVAL_CUTOFF]
    if DE_LFC_COL in up_df.columns:
        up_df = up_df[up_df[DE_LFC_COL] > VOLCANO_LFC_CUTOFF]
    if DE_PVAL_COL in up_df.columns:
        up_df = up_df.sort_values(DE_PVAL_COL)
    if up_df.empty:
        return

    tbl_top = _add_table_section_title(slide, left, top, width, "TOP UPREGULATED GENES")

    preferred = [GENE_SYMBOL_COL, DE_LFC_COL, DE_PVAL_COL, "baseMean", "gene_id"]
    show_cols = [c for c in preferred if c in up_df.columns][:5]
    if not show_cols:
        show_cols = list(up_df.columns[:5])

    sub = up_df[show_cols].head(PPTX_TOP_DE_GENES).fillna("").copy()
    _pval_cols = {DE_PVAL_COL, "padj", "pvalue", "pvals_adj", "pvals"}
    for c in sub.select_dtypes(include="float").columns:
        if c in _pval_cols:
            sub[c] = sub[c].apply(
                lambda v: f"{v:.2e}" if pd.notna(v) and v != "" else ""
            )
        else:
            sub[c] = sub[c].round(4).astype(str)

    n_rows = len(sub) + 1   # +1 header row
    n_cols = len(show_cols)
    row_h  = Cm(0.42)
    tbl    = slide.shapes.add_table(n_rows, n_cols, left, tbl_top, width, row_h * n_rows).table

    col_w = width // n_cols
    for ci in range(n_cols):
        tbl.columns[ci].width = col_w

    for ci, col in enumerate(show_cols):
        _set_cell_text(tbl.cell(0, ci), col, bold=True, fontsize=6, bg_color=_LGREY)

    for ri, (_, row) in enumerate(sub.iterrows(), start=1):
        for ci, col in enumerate(show_cols):
            _set_cell_text(tbl.cell(ri, ci), str(row[col]), fontsize=6)


# ── Build presentation ───────────────────────────────────────────────────────
prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H
blank_layout = prs.slide_layouts[6]   # blank

for card in subcluster_data:
    slide = prs.slides.add_slide(blank_layout)

    _add_title_bar(slide, card["label"], card["disease"])

    # ── Left panel: stats table then DE table ─────────────────────────────
    stats_bottom = _add_stats_table(slide, card["stats"], card["disease_breakdown"])
    _add_de_table(slide, card.get("de_df"), top=stats_bottom + Cm(1.2))

    # ── Right panel: images ───────────────────────────────────────────────
    # Row 1: 3× UMAP — taller and wider to fill available space
    umap_top = Cm(2.0)
    umap_h   = Cm(7.2)
    umap_w   = Cm(6.5)
    gap      = Cm(0.18)

    for i, key in enumerate(["umap_celltype_bytes", "umap_disease_bytes", "umap_highlight_bytes"]):
        _add_image_bytes(slide, card.get(key),
                         _RIGHT_L + i * (umap_w + gap), umap_top, umap_w, umap_h)

    # Row 2: volcano + dotplot — stretch to fill remaining vertical space
    plot_top = umap_top + umap_h + Cm(0.25)
    plot_h   = SLIDE_H - plot_top - Cm(0.25)   # fill to bottom margin
    plot_w   = Cm(10.0)

    _add_image_bytes(slide, card.get("volcano_bytes"),
                     _RIGHT_L,              plot_top, plot_w, plot_h)
    _add_image_bytes(slide, card.get("dotplot_bytes"),
                     _RIGHT_L + plot_w + gap, plot_top, plot_w, plot_h)

pptx_path = OUTPUT_DIR / f"{TISSUE}_subcluster_cards.pptx"
prs.save(str(pptx_path))
print(f"✓ PowerPoint deck written: {pptx_path}  ({pptx_path.stat().st_size / 1024:.0f} KB)")


## Summary

| Output | Path |
|--------|------|
| HTML report | `{REPORT_DIR}/index.html` |
| Report images | `{REPORT_DIR}/img/` |
| PowerPoint  | `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` |

**Next step:** open `{REPORT_DIR}/index.html` in a browser (the `img/` folder must stay alongside `index.html`).  
Share the `.pptx` or zip the entire `{REPORT_DIR}/` directory with collaborators for review.  
Once feedback is received, update `INCLUDED_SUBCLUSTERS` in `02_consensus_gene_list.ipynb`  
to restrict the consensus gene list to approved subclusters only.
